# Evaluate the best checkpoint under three input conditions
Select the **symile-drfuse** kernel and run the setup cells first. Then run each of the three evaluation cells below independently, or use **Run All**. This notebook performs **testing only** and does not train or change weights.

Each condition uses the **same checkpoint and final prediction head**, the same 464 clinical test admissions, and the same known label entries. Excluded modality arrays are not opened by that dataset view and no zero placeholders are created. AP means average precision; undefined scores appear as NaN and are excluded from macro averages. AP=1 on a positive-only finding does not demonstrate discrimination from negatives.

In [1]:
from argparse import Namespace
from pathlib import Path
import sys

import lightning.pytorch as pl
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

# Locate Pipeline2 from Research, Pipeline2, or this notebook's directory.
PROJECT_DIR = next(
    folder if (folder / "models/main_trainer.py").is_file() else folder / "Pipeline2"
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "models/main_trainer.py").is_file()
    or (folder / "Pipeline2/models/main_trainer.py").is_file()
)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from models import MainTrainer
from utils import TARGETS, TEST_CONDITIONS, SymileMIMICDataset

## Select the best checkpoint
The default below is the completed `version_3` run. Change `CHECKPOINT_PATH` to review another run. Use the actual best filename if Lightning adds a suffix such as `best-v1.ckpt`. Architecture settings are read from the checkpoint, so they do not need to be entered again.

In [2]:
CHECKPOINT_PATH = PROJECT_DIR / "runs_complete/version_3/checkpoints/best.ckpt"
DATASET_ROOT = PROJECT_DIR.parent / "symile-mimic/symile-mimic-aws"
BATCH_SIZE = 8

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Set CHECKPOINT_PATH to your saved best checkpoint: {CHECKPOINT_PATH}")

# Load only your own trusted checkpoint. Keep optimizer tensors memory-mapped on CPU.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False, mmap=True)
hparams = dict(checkpoint["hyper_parameters"])
if hparams.get("training_protocol") != "complete_inputs_three_tests_v1":
    raise ValueError("Choose a checkpoint from the complete-input training protocol.")
label_names = list(hparams.pop("label_names"))
if label_names != TARGETS:
    raise ValueError("The checkpoint target names or order differ from our 14 targets.")

pl.seed_everything(hparams["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

module = MainTrainer(args=Namespace(**hparams), label_names=label_names)
module.load_state_dict(checkpoint["state_dict"], strict=True)
module.eval()
module.requires_grad_(False)
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch index:", checkpoint["epoch"])
print("Completed epochs at checkpoint:", checkpoint["epoch"] + 1)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
del checkpoint

# No logger, new training run directory, checkpoint writes, or optimizer updates.
evaluator = pl.Trainer(
    accelerator="auto", devices=1,
    logger=False, enable_checkpointing=False, enable_model_summary=False,
)
results = {}

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Checkpoint: /home/rittick-dutta/Desktop/Research/Pipeline2/runs_complete/version_3/checkpoints/best.ckpt
Checkpoint epoch index: 15
Completed epochs at checkpoint: 16
Device: CUDA


## Shared evaluation helper
This calls the existing trainer's test code, so metric calculations and label masking match `main.py`. Every evaluation displays a macro-score table and a separate 14-row finding table. It also retains admission-level probabilities in the returned predictions DataFrame.

In [3]:
def evaluate_condition(condition):
    # Create only this input view; excluded modality files are never opened here.
    modalities = TEST_CONDITIONS[condition]
    dataset = SymileMIMICDataset(DATASET_ROOT, split="test", modalities=modalities)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    evaluator.test(model=module, dataloaders=loader, verbose=False)
    result = module.test_results

    # Ensure every independently run cell uses the same ordered test cohort.
    if results:
        reference = next(iter(results.values()))
        for key in ["hadm_ids", "y_gt", "label_mask"]:
            if not torch.equal(result[key], reference[key]):
                raise ValueError(f"Test conditions differ in {key}.")
    results[condition] = result

    summary = pd.DataFrame([{
        "condition": condition,
        "modalities": "+".join(modalities),
        "admissions": len(dataset),
        "macro_AP": result["ap"],
        "macro_AUROC": result["auroc"],
        "AP_findings": int(result["mlaps"].isfinite().sum()),
        "AUROC_findings": int(result["mlroc"].isfinite().sum()),
    }])
    findings = dataset.label_counts()
    findings.insert(1, "AP", result["mlaps"].numpy())
    findings.insert(2, "AUROC", result["mlroc"].numpy())
    predictions = pd.DataFrame(result["preds"].numpy(), columns=label_names)
    predictions.insert(0, "hadm_id", result["hadm_ids"].numpy())
    print("Checkpoint:", CHECKPOINT_PATH.name, "| Inputs:", modalities)
    print("Attention tokens:", result["attn_token_names"])
    return result, summary, findings, predictions

## Test 1 All modalities
CXR + ECG + labs. Attention uses the three distinct embeddings and one fused shared embedding.

In [4]:
all_result, all_summary, all_findings, all_predictions = evaluate_condition("all")
display(all_summary.round(4))
display(all_findings.round(4))
# all_predictions contains each admission's 14 final probabilities.

You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/rittick-dutta/anaconda3/envs/symile-drfuse/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

Checkpoint: best.ckpt | Inputs: ('cxr', 'ecg', 'labs')
Attention tokens: ('ecg_distinct', 'labs_distinct', 'shared', 'cxr_distinct')


,condition,modalities,admissions,macro_AP,macro_AUROC,AP_findings,AUROC_findings
0,all,cxr+ecg+labs,464,0.8631,0.7477,14,11


,finding,AP,AUROC,positive,negative,uncertain,unreported
0,Atelectasis,0.9834,0.6908,141,5,22,296
1,Cardiomegaly,0.9145,0.7981,144,50,20,250
2,Consolidation,0.8224,0.7931,29,22,8,405
3,Edema,0.8353,0.8444,82,88,39,255
4,Enlarged Cardiomediastinum,0.8109,0.6690,28,15,30,391
5,Fracture,1.0000,NaN,10,0,0,454
6,Lung Lesion,1.0000,1.0000,12,2,3,447
7,Lung Opacity,0.9722,0.7933,131,13,12,308
8,No Finding,1.0000,NaN,69,0,0,395
9,Pleural Effusion,0.8879,0.8131,166,90,15,193


## Test 2 CXR only
ECG and labs are excluded. Attention uses CXR distinct and CXR shared; predictions still come from the same final head, not the CXR auxiliary head.

In [5]:
cxr_result, cxr_summary, cxr_findings, cxr_predictions = evaluate_condition("cxr_only")
display(cxr_summary.round(4))
display(cxr_findings.round(4))
# cxr_predictions contains each admission's 14 final probabilities.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/rittick-dutta/anaconda3/envs/symile-drfuse/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

Checkpoint: best.ckpt | Inputs: ('cxr',)
Attention tokens: ('shared', 'cxr_distinct')


,condition,modalities,admissions,macro_AP,macro_AUROC,AP_findings,AUROC_findings
0,cxr_only,cxr,464,0.8609,0.7278,14,11


,finding,AP,AUROC,positive,negative,uncertain,unreported
0,Atelectasis,0.9795,0.6482,141,5,22,296
1,Cardiomegaly,0.9167,0.7919,144,50,20,250
2,Consolidation,0.8025,0.7900,29,22,8,405
3,Edema,0.8258,0.8307,82,88,39,255
4,Enlarged Cardiomediastinum,0.8166,0.6619,28,15,30,391
5,Fracture,1.0000,NaN,10,0,0,454
6,Lung Lesion,1.0000,1.0000,12,2,3,447
7,Lung Opacity,0.9701,0.7827,131,13,12,308
8,No Finding,1.0000,NaN,69,0,0,395
9,Pleural Effusion,0.8882,0.8177,166,90,15,193


## Test 3 ECG and labs
CXR is excluded. Attention uses ECG distinct, labs distinct, and their fused shared embedding.

In [6]:
ecg_labs_result, ecg_labs_summary, ecg_labs_findings, ecg_labs_predictions = evaluate_condition("ecg_labs")
display(ecg_labs_summary.round(4))
display(ecg_labs_findings.round(4))
# ecg_labs_predictions contains each admission's 14 final probabilities.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/rittick-dutta/anaconda3/envs/symile-drfuse/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

Checkpoint: best.ckpt | Inputs: ('ecg', 'labs')
Attention tokens: ('ecg_distinct', 'labs_distinct', 'shared')


,condition,modalities,admissions,macro_AP,macro_AUROC,AP_findings,AUROC_findings
0,ecg_labs,ecg+labs,464,0.8115,0.6343,14,11


,finding,AP,AUROC,positive,negative,uncertain,unreported
0,Atelectasis,0.9916,0.7872,141,5,22,296
1,Cardiomegaly,0.8101,0.6264,144,50,20,250
2,Consolidation,0.8030,0.7053,29,22,8,405
3,Edema,0.6585,0.6912,82,88,39,255
4,Enlarged Cardiomediastinum,0.7211,0.5643,28,15,30,391
5,Fracture,1.0000,NaN,10,0,0,454
6,Lung Lesion,0.9253,0.6667,12,2,3,447
7,Lung Opacity,0.9275,0.5291,131,13,12,308
8,No Finding,1.0000,NaN,69,0,0,395
9,Pleural Effusion,0.7773,0.6722,166,90,15,193
